In [2]:
import os
import certifi

from dotenv import load_dotenv

from langchain_google_genai import GoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests


/opt/homebrew/Caskroom/miniconda/base/envs/langagent/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
from langchain.agents import create_react_agent, AgentExecutor

In [3]:
#  Load ENV

os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
W_STACK= os.getenv("W_STACK")

In [4]:
print(bool(os.getenv("W_STACK")))

True


In [30]:
search_tool = TavilySearchResults(max_results=3)

In [43]:
@tool
def get_weather(city: str) -> str:
    """
    Fetch current weather
    """

    url = (
        f"https://weatherstack.com/current?"
        f"access_key={W_STACK}&query={city}"
    )

    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']} celsius"
    )


In [53]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [ ]:
prompt = hub.pull("hwchase17/react")

/opt/homebrew/Caskroom/miniconda/base/envs/langagent/lib/python3.11/site-packages/langchain/hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [55]:
# Tools

tools = [search_tool, get_weather]

In [56]:
# Create agent

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [57]:
# Executor

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True

)

In [ ]:
# Run 

response = agent_executor.invoke({
    "input": (
        "Find the capital of India and it's current weather"
        
    )
})

print(response["output"])

In [ ]:
import os
import certifi
import requests

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_tool_calling_agent, AgentExecutor

os.environ["SSL_CERT_FILE"] = certifi.where()

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
W_STACK = os.getenv("W_STACK")

search_tool = TavilySearchResults(max_results=3)


@tool
def get_weather(city: str) -> str:
    """Fetch current weather for a city."""
    response = requests.get(
        "http://api.weatherstack.com/current",
        params={"access_key": W_STACK, "query": city},
        timeout=15,
    )
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather for {city}: {data.get('error', {}).get('info', 'unknown error')}"

    return f"City: {city}\nTemperature: {data['current']['temperature']} Celsius"


llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the available tools whenever you need current information."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

tools = [search_tool, get_weather]

agent = create_tool_calling_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)

response = agent_executor.invoke({
    "input": "Find the capital of India and its current weather"
})

print(response["output"])

In [7]:
from importlib.metadata import version

print("langchain-groq:", version("langchain-groq"))
print("groq:", version("groq"))
print("langchain:", version("langchain"))
print("langchain-core:", version("langchain-core"))

langchain-groq: 1.1.3
groq: 0.37.1
langchain: 0.1.16
langchain-core: 1.6.5


In [ ]:
import os
import certifi
import requests

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_tool_calling_agent, AgentExecutor

os.environ["SSL_CERT_FILE"] = certifi.where()

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
W_STACK = os.getenv("W_STACK")

search_tool = TavilySearchResults(max_results=3)


@tool
def get_weather(city: str) -> str:
    """Fetch current weather for a city."""
    response = requests.get(
        "http://api.weatherstack.com/current",
        params={"access_key": W_STACK, "query": city},
        timeout=15,
    )
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather for {city}: {data.get('error', {}).get('info', 'unknown error')}"

    return f"City: {city}\nTemperature: {data['current']['temperature']} Celsius"


llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the available tools whenever you need current information."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

tools = [search_tool, get_weather]

agent = create_tool_calling_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)

response = agent_executor.invoke({
    "input": "Find the capital of India and its current weather"
})

print(response["output"])

In [ ]:
from importlib.metadata import version

print("langchain-groq:", version("langchain-groq"))
print("groq:", version("groq"))
print("langchain:", version("langchain"))
print("langchain-core:", version("langchain-core"))

In [1]:
from importlib.metadata import version

print("langchain-groq:", version("langchain-groq"))
print("groq:", version("groq"))
print("langchain:", version("langchain"))
print("langchain-core:", version("langchain-core"))

langchain-groq: 0.1.5
groq: 0.5.0
langchain: 0.1.16
langchain-core: 0.1.45


In [1]:
import os
import certifi
import requests

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_tool_calling_agent, AgentExecutor

os.environ["SSL_CERT_FILE"] = certifi.where()

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
W_STACK = os.getenv("W_STACK")

search_tool = TavilySearchResults(max_results=3)


@tool
def get_weather(city: str) -> str:
    """Fetch current weather for a city."""
    response = requests.get(
        "http://api.weatherstack.com/current",
        params={"access_key": W_STACK, "query": city},
        timeout=15,
    )
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather for {city}: {data.get('error', {}).get('info', 'unknown error')}"

    return f"City: {city}\nTemperature: {data['current']['temperature']} Celsius"


llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY,
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the available tools whenever you need current information."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

tools = [search_tool, get_weather]

agent = create_tool_calling_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)

response = agent_executor.invoke({
    "input": "Find the capital of India and its current weather"
})

print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'capital of India'}`


[{'url': 'https://timesofindia.indiatimes.com/india/list-of-cities-that-served-as-the-capital-of-india/articleshow/111815680.cms', 'content': "| Era | Capital(s) | Description |\n| Modern | Allahabad | Capital of India for a day in 1858, capital of North-Western Provinces. |\n| Calcutta | Capital of British India until 1911. |\n| Shimla | Summer capital by the latter half of the 19th century. |\n| New Delhi | Capital of India since 1911, buildings inaugurated in early 1931. |\n| Mediaeval | Agra | Capital of Sikandar Lodi during the Lodhi dynasty, moved from Delhi in 1506. |\n| Vijayanagara | Capital of Vijayanagara Empire until 1585. |\n| Patna | Capital of Sher Shah Suri's Empire (1538/1540-1556) and Bihar Subah under Mughals. |\n| Allahabad | Provincial capital in the Mughal Empire, Jahangir's headquarters (1599-1604). |\n| Ghor | Capital of the Ghurid Sultanate. |\